![Teste vocacional com Machine Learning](capa.png)


# Projeto - Teste Vocacional com Machine Learning
## Etapa 01 - Coleta de Dados

## O Problema de Negócio

Escolher um curso superior é uma das decisões mais importantes na vida de um jovem, e muitas vezes ela é tomada com pouca informação: por pressão familiar, pela fama da profissão ou por influência dos amigos. O resultado aparece nos números: uma parcela grande dos estudantes troca de curso ou abandona a graduação nos primeiros anos.

Este projeto propõe um **teste vocacional baseado em dados**, que usa Machine Learning para relacionar o **perfil de interesses** de uma pessoa às **áreas de formação** que mais combinam com ela.

## Objetivo

Responder à pergunta: **"dado o perfil de interesses de uma pessoa, quais áreas de formação combinam mais com ela?"**

O sistema recomendará as **3 áreas mais compatíveis** e explicará quais características do perfil pesaram em cada recomendação.

## Fundamentação: o modelo RIASEC de Holland

O projeto usa o **modelo RIASEC**, criado pelo psicólogo John Holland e amplamente adotado em orientação vocacional. Ele classifica os interesses em seis dimensões:

| Sigla | Dimensão | Perfil |
|---|---|---|
| **R** | Realista | gosta de atividades práticas, ferramentas, máquinas, natureza |
| **I** | Investigativo | gosta de pesquisar, analisar, resolver problemas |
| **A** | Artístico | gosta de criar, se expressar, inovar |
| **S** | Social | gosta de ajudar, ensinar, cuidar de pessoas |
| **E** | Empreendedor | gosta de liderar, persuadir, negociar |
| **C** | Convencional | gosta de organização, dados, rotinas e processos |

## Fontes de Dados

- **Open Psychometrics (RIASEC, 48 itens):** respostas públicas de milhares de pessoas ao teste RIASEC, incluindo o curso/formação de cada uma. É a **base de treino** do modelo.
- **INEP - Censo da Educação Superior e classificação CINE Brasil:** contexto brasileiro (cursos mais procurados, concluintes) e padronização das áreas de formação.
- **MEC - Catálogo de Cursos e Diretrizes Curriculares:** descrição do que cada curso estuda.

## Etapas do Projeto (Pipeline de Ciência de Dados)

1. Coleta de dados *(este notebook)*
2. Limpeza e engenharia de atributos
3. Análise exploratória dos dados (EDA)
4. Modelagem
5. Avaliação do modelo
6. Deploy da aplicação web

## Considerações Éticas

- O sistema **sugere, não determina**: o resultado é um ponto de partida para reflexão, não uma sentença sobre o futuro de ninguém.
- Não são coletados dados pessoais identificáveis, em respeito à **LGPD**.
- **Limitação conhecida:** a base de treino tem respondentes majoritariamente de países de língua inglesa, o que pode introduzir viés. Esse ponto será analisado e documentado ao longo do projeto.

In [80]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python usada neste Projeto:', python_version())


Versão da Linguagem Python usada neste Projeto: 3.14.4


In [81]:
# Os pacotes deste projeto foram instalados no ambiente virtual (.venv),
# pelo terminal do VS Code, com o ambiente ativado:
#
#   .venv\Scripts\Activate.ps1
#   pip install pandas numpy matplotlib seaborn ipykernel watermark
#
# Para atualizar um pacote:
#   pip install -U nome_pacote
#
# Para instalar uma versão exata de um pacote:
#   pip install nome_pacote==versão_desejada
#
# Depois de instalar ou atualizar um pacote, reinicie o kernel do notebook.


### Importando as bibliotecas necessárias


In [ ]:
# Para manipulação dos dados - a dupla inseparável
import pandas as pd
import numpy as np

# Permite importar os módulos da pasta src/ do projeto
import sys
sys.path.append('../src')

# Funções de apoio escritas para o projeto (src/limpeza.py)
from limpeza import calcular_porcentagem_valores_ausentes, obter_porcentagem_valores_ausentes

In [83]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Gustavo" --iversions


Author: Gustavo

numpy   : 2.5.3
pandas  : 3.0.6
platform: 1.0.9



### Importação do conjunto de dados

In [84]:
# Importando o dataset RIASEC (Open Psychometrics)
df = pd.read_csv('../dados/Dados originais/data.csv', sep='\t')
df.head(10)


C:\Users\Gusta\AppData\Local\Temp\ipykernel_93140\773830837.py:2: DtypeWarning: Columns (93: Unnamed: 93) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../dados/Dados originais/data.csv', sep='\t')


,R1,R2,R3,R4,R5,R6,R7,R8,I1,I2,...,orientation,race,voted,married,familysize,uniqueNetworkLocation,country,source,major,Unnamed: 93
0,3,4,3,1,1,4,1,3,5,5,...,1,1,2,1,1,1,US,2,NaN,NaN
1,1,1,2,4,1,2,2,1,5,5,...,3,4,1,2,3,1,US,1,Nursing,NaN
2,2,1,1,1,1,1,1,1,4,1,...,1,4,2,1,1,1,US,1,NaN,NaN
3,3,1,1,2,2,2,2,2,4,1,...,1,1,2,1,1,1,CN,0,NaN,NaN
4,4,1,1,2,1,1,1,2,5,5,...,3,1,2,1,4,1,PH,0,education,NaN
5,3,5,1,3,1,5,3,4,4,5,...,1,5,1,1,2,1,IN,2,Literature,NaN
6,1,4,1,4,1,4,1,2,4,4,...,1,1,2,1,3,2,US,0,Math,NaN
7,5,1,2,2,2,1,2,1,4,4,...,3,1,2,1,0,1,PH,0,mathematics and science,NaN
8,1,1,1,1,1,1,1,1,2,1,...,1,1,1,1,3,1,US,0,family resource management,NaN
9,4,2,2,2,2,2,3,2,2,2,...,1,1,2,1,6,1,MY,1,Counselling,NaN


Já nas primeiras linhas é possível notar valores nulos (`NaN`) na coluna `major` (curso) e uma coluna sem nome no final, `Unnamed: 93`, que não existe no codebook.

In [85]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145828 entries, 0 to 145827
Data columns (total 94 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   R1                     145828 non-null  int64
 1   R2                     145828 non-null  int64
 2   R3                     145828 non-null  int64
 3   R4                     145828 non-null  int64
 4   R5                     145828 non-null  int64
 5   R6                     145828 non-null  int64
 6   R7                     145828 non-null  int64
 7   R8                     145828 non-null  int64
 8   I1                     145828 non-null  int64
 9   I2                     145828 non-null  int64
 10  I3                     145828 non-null  int64
 11  I4                     145828 non-null  int64
 12  I5                     145828 non-null  int64
 13  I6                     145828 non-null  int64
 14  I7                     145828 non-null  int64
 15  I8                     14582

### Estatísticas descritivas (média, mínimo, máximo...)

In [ ]:
# Estatísticas descritivas das colunas numéricas
df.describe()

In [ ]:
# Número de linhas e colunas do dataset
df.shape

### Verificando valores ausentes

As funções usadas abaixo estão no módulo `src/limpeza.py`:

- `calcular_porcentagem_valores_ausentes`: **exibe** a porcentagem de células vazias do dataset (usa `print`).
- `obter_porcentagem_valores_ausentes`: **devolve** a porcentagem (usa `return`), permitindo usar o valor em outras partes do código, como no `if` abaixo.

In [89]:
# Verificando a porcentagem de valores ausentes no dataset
calcular_porcentagem_valores_ausentes(df)


O conjunto de dados tem 1.45% de valores ausentes.


In [ ]:
# Guardando o resultado da função numa variável
porcentagem = obter_porcentagem_valores_ausentes(df)

# Usando o resultado para tomar uma decisão
if porcentagem > 5:
    print(f"Atenção: {porcentagem:.2f}% de ausentes, precisamos tratar!")
else:
    print(f"Apenas {porcentagem:.2f}% de ausentes, dataset em bom estado.")

### Investigando a coluna fantasma

No `df.info()`, a última coluna chama atenção:

```
93  Unnamed: 93        1 non-null       str
```

Ela deveria estar totalmente vazia, mas possui **um único valor**, que precisa ser analisado antes de qualquer decisão.

In [ ]:
# Filtrando apenas a linha que contém valor na coluna Unnamed: 93
linha_fantasma = df[df['Unnamed: 93'].notna()]

# Exibindo a linha completa, transposta, para caber na tela
linha_fantasma.T

Trata-se de um participante russo que cursou economia. Provavelmente ele digitou um TAB antes do nome do curso. Como o arquivo usa TAB como separador, o valor *economics* foi deslocado uma coluna para a direita e o campo `major` ficou vazio.

Esse valor será resgatado para a coluna do curso na etapa de limpeza (`02_limpeza.ipynb`).

---
## Conclusões da exploração inicial

- O dataset possui **145.828 participantes** e **94 colunas**, separadas por TAB.
- As colunas se dividem em: 48 perguntas RIASEC (`R1` a `C8`), tempos de resposta, 10 itens de personalidade (`TIPI`), 16 itens de vocabulário (`VCL`), perguntas demográficas e o curso (`major`). O significado de cada uma está em `docs/codebook_ptbr.md`.
- Apenas **1,45%** das células estão vazias, mas quase tudo se concentra em duas colunas: `major` (cerca de 36% dos participantes não informaram o curso) e a coluna fantasma `Unnamed: 93`.
- A coluna `Unnamed: 93` foi criada pelo TAB extra no fim de cada linha, e guarda um único curso válido, deslocado por um erro de digitação.
- As respostas RIASEC vão de **0 a 5**, mas o codebook define a escala de 1 a 5: o **0 indica pergunta não respondida**.
- `age` e `familysize` têm valores máximos impossíveis (2.147.483.647, o maior inteiro de 32 bits), indicando respostas inválidas.
- O codebook informa que as palavras de `VCL6`, `VCL9` e `VCL12` **não existem** e servem para detectar respostas desatentas.

Todos esses pontos são tratados no notebook `02_limpeza.ipynb`.

## Fim da Etapa 01